In [ ]:
"""
Notebook 06 v3 — Experiments & Robustness Assessment

Purpose:
    Systematic evaluation of the main model under a range of controlled
    conditions that go beyond the random 70/15/15 split of Notebook 05.

Experiments:
    1. Random Stratified 5-Fold CV — variance estimation of the baseline
    2. Spatial split — train/val on west half of ROI, test on east half
    3. Temporal split — window-based, evaluates cross-time transfer
    4. Combined spatial+temporal — the most demanding evaluation
    5. Random Forest baselines — RF on raw and PCA-reduced features
    6. L-TAE on PCA features — verifies use of the full feature space
    7. Architecture ablations — positional encoding, attention heads, MLP
    8. Deployment realism — reweight metrics to natural ROI distribution

What changes from v2:
    * Every trained model uses MaskedMultiTaskLoss
    * Each experiment additionally reports a per-land-cover breakdown

Outputs:
    models/experiments/<exp_name>/results.json for each experiment
    outputs/tables/experiment_summary.csv for the aggregated comparison
"""

# Notebook 06 v3 — Experiments

In [ ]:
# --- Environment auto-detection ---
import sys, os
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    THESIS_ROOT = "/content/drive/MyDrive/thesisv4"
else:
    THESIS_ROOT = os.path.expanduser("~/thesisv4")

sys.path.insert(0, THESIS_ROOT)
print(f"=== Environment: {'COLAB' if IS_COLAB else 'LOCAL'} ===")
print(f"Workspace: {THESIS_ROOT}")

In [ ]:
import sys, importlib
# sys.path already set by auto-detect cell above
import config, model_defs, metrics
for m in (config, model_defs, metrics):
    importlib.reload(m)

config.ensure_dirs()
config.set_seed()
config.print_summary()

In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold

DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
EXP_ROOT = config.PATHS["models_exp"]
print(f"Device: {DEVICE}")
print(f"Experiments root: {EXP_ROOT}")

## 0. Load shared data and helper trainer

In [ ]:
X        = np.load(config.FILES["X_scaled"])
y        = np.load(config.FILES["y_burned"])
mask     = np.load(config.FILES["X_scaled"].replace(".npy", "_mask.npy"))
targets  = dict(np.load(config.FILES["targets_aux"]))
meta_df  = pd.read_csv(
    config.FILES["X_scaled"].replace(".npy", "_meta.csv"))
coords_df = pd.read_csv(config.FILES["pixel_coords"])

with open(config.FILES["feature_names"]) as f:
    feature_names = json.load(f)

N, T, C = X.shape
lc_labels_all = meta_df["lc_dominant"].astype(int).values
print(f"X shape: {X.shape}")
print(f"Features: {C}")
print(f"Total pixels: {N}")

In [ ]:
# ---------------------------------------------------------------
# Shared per-epoch progress line, used by every experiment below so
# a long run always shows where it is instead of appearing frozen.
# ---------------------------------------------------------------
def log_epoch(tag, epoch, n_epochs, val_m, secs, t0, every=1):
    """Print one progress line. `every` thins the output if wanted."""
    if not (epoch == 1 or epoch == n_epochs or epoch % every == 0):
        return
    done = time.time() - t0
    eta = done / epoch * (n_epochs - epoch)
    print(f"{tag}, epoch {epoch}/{n_epochs} - {secs:.1f} s/epoch, "
          f"val_auc {val_m['auc']:.4f}, val_loss {val_m['total']:.4f}"
          f"  [elapsed {done/60:.1f}m, eta {eta/60:.1f}m]", flush=True)


def train_and_evaluate(idx_train, idx_val, idx_test,
                       exp_name, exp_dir,
                       targets_override=None,
                       mask_override=None,
                       n_features_override=None,
                       epochs=None,
                       progress=None):
    """Train a fresh L-TAE model on the supplied splits and evaluate it.

    Optional overrides let callers swap in temporal-split targets (recomputed
    for the truncated time window) or feature-reduced inputs (PCA) without
    duplicating the full training loop.
    """
    config.set_seed()

    tgt = targets_override if targets_override is not None else targets
    msk = mask_override if mask_override is not None else mask
    n_feats = n_features_override if n_features_override is not None else C
    n_epochs = epochs if epochs is not None else config.TRAINING["epochs"]

    train_loader = metrics.make_loader(X, y, msk, tgt, idx_train,
                                       config.TRAINING["batch_size"], True)
    val_loader   = metrics.make_loader(X, y, msk, tgt, idx_val,
                                       config.TRAINING["batch_size"], False)
    test_loader  = metrics.make_loader(X, y, msk, tgt, idx_test,
                                       config.TRAINING["batch_size"], False)

    model = model_defs.FireRecoveryModel(
        n_features=n_feats,
        d_model=config.MODEL["d_model"],
        n_head=config.MODEL["n_head"],
        d_k=config.MODEL["d_k"],
        max_len=T,
        tau=config.MODEL["positional_tau"],
        dropout=config.MODEL["dropout"],
    ).to(DEVICE)

    criterion = model_defs.MaskedMultiTaskLoss(
        alpha=config.LOSS["focal_alpha"],
        gamma=config.LOSS["focal_gamma"],
        lambda_aux=config.LOSS["lambda_aux"],
    )
    optimizer = optim.AdamW(model.parameters(),
                            lr=config.TRAINING["lr"],
                            weight_decay=config.TRAINING["weight_decay"])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=n_epochs)

    history = {"train": [], "val": []}
    best_val_auc = -1.0
    best_state   = None

    for epoch in range(1, n_epochs + 1):
        _t0 = time.time()
        # compute_auc=False on the training pass: the AUC over every training
        # timestep costs ~13.6 s/epoch, more than the training step itself,
        # and is only written to history.json. Model selection below and every
        # plot use the validation AUC, which is still computed.
        train_m = metrics.run_epoch(model, criterion, optimizer,
                                    train_loader, DEVICE, train=True,
                                    grad_clip=config.TRAINING["grad_clip"],
                                    compute_auc=False)
        val_m   = metrics.run_epoch(model, criterion, optimizer,
                                    val_loader, DEVICE, train=False)
        scheduler.step()
        history["train"].append(train_m)
        history["val"].append(val_m)

        if val_m["auc"] > best_val_auc:
            best_val_auc = val_m["auc"]
            best_state   = {k: v.cpu().clone()
                            for k, v in model.state_dict().items()}

        if progress is not None:
            progress(epoch, n_epochs, train_m, val_m, time.time() - _t0)

    model.load_state_dict(best_state)
    best_thr, best_f1, _, _, _ = metrics.tune_threshold(
        model, val_loader, DEVICE)
    burned_metrics, correlations, predictions = metrics.evaluate_test_set(
        model, test_loader, DEVICE, best_thr, tgt, idx_test)
    per_category = metrics.per_category_stats(
        predictions, idx_test, meta_df, config.PIXEL_TYPES)
    per_lc = metrics.per_lc_stats(
        predictions, idx_test, lc_labels_all, config.LC_CATEGORIES)

    n_params = sum(p.numel() for p in model.parameters())
    extra = {
        "training":   {"best_val_auc": float(best_val_auc),
                       "n_epochs": n_epochs},
        "data_info":  {"n_train": len(idx_train),
                       "n_val":   len(idx_val),
                       "n_test":  len(idx_test)},
        "model_info": {"n_parameters": n_params,
                       "loss_type":    "MaskedMultiTaskLoss"},
        "per_lc":     per_lc,
    }
    metrics.save_experiment_results(
        exp_dir, exp_name, best_state, history,
        burned_metrics, correlations, per_category, extra=extra)
    metrics.print_summary(exp_name, n_params, burned_metrics,
                          correlations, per_category, config.PIXEL_TYPES)
    metrics.print_lc_summary(per_lc, config.LC_CATEGORIES)
    return {
        "burned_metrics": burned_metrics,
        "correlations":   correlations,
        "per_category":   per_category,
        "per_lc":         per_lc,
        "best_val_auc":   best_val_auc,
    }

## Experiment 1 — Random Stratified 5-Fold CV

Estimates the variance of the random-split baseline. Each fold: 80%
train+val, 20% test; within the 80%, ~88%/12% train/val to match the
main model's ~85%/15% test proportion.

In [ ]:
EXP_NAME = "exp1_random_kfold"
EXP_DIR  = os.path.join(EXP_ROOT, EXP_NAME)
os.makedirs(EXP_DIR, exist_ok=True)

strat = meta_df["pixel_type"].apply(
    lambda t: "no_fires" if t == "no_fires" else "fires").values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=config.SEED)
fold_results = []

for fold_idx, (trainval_idx, test_idx) in enumerate(
        skf.split(np.arange(N), strat)):

    rng = np.random.RandomState(config.SEED + fold_idx)
    rng.shuffle(trainval_idx)
    n_val = len(trainval_idx) // 8
    fold_val_idx   = trainval_idx[:n_val]
    fold_train_idx = trainval_idx[n_val:]

    print(f"\n=== Fold {fold_idx} ===")
    print(f"Train: {len(fold_train_idx)}  Val: {len(fold_val_idx)}  "
          f"Test: {len(test_idx)}")

    fold_dir = os.path.join(EXP_DIR, f"fold_{fold_idx}")

    _fold_t0 = time.time()

    def _progress(epoch, n_epochs, train_m, val_m, secs, _f=fold_idx,
                  _t0=_fold_t0):
        log_epoch(f"Fold {_f}", epoch, n_epochs, val_m, secs, _t0)

    res = train_and_evaluate(
        fold_train_idx, fold_val_idx, test_idx,
        exp_name=f"{EXP_NAME}_fold{fold_idx}",
        exp_dir=fold_dir,
        progress=_progress,
    )
    print(f"=== fold {fold_idx} done in "
          f"{(time.time()-_fold_t0)/60:.1f} min ===", flush=True)
    fold_results.append(res)

In [ ]:
print("=" * 70)
print("EXPERIMENT 1 — Random 5-Fold CV: aggregated results")
print("=" * 70)

aucs     = [r["burned_metrics"]["auc"] for r in fold_results]
f1s      = [r["burned_metrics"]["f1"]  for r in fold_results]
sev_pear = [r["correlations"]["severity"]["pearson"]     for r in fold_results]
per_pear = [r["correlations"]["persistence"]["pearson"]  for r in fold_results]
rec_pear = [r["correlations"]["recovery_gap"]["pearson"] for r in fold_results]

summary = {
    "auc":                  {"mean": float(np.mean(aucs)),     "std": float(np.std(aucs))},
    "f1":                   {"mean": float(np.mean(f1s)),      "std": float(np.std(f1s))},
    "severity_pearson":     {"mean": float(np.mean(sev_pear)), "std": float(np.std(sev_pear))},
    "persistence_pearson":  {"mean": float(np.mean(per_pear)), "std": float(np.std(per_pear))},
    "recovery_gap_pearson": {"mean": float(np.mean(rec_pear)), "std": float(np.std(rec_pear))},
}
with open(os.path.join(EXP_DIR, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print(f"\nAUC          : {summary['auc']['mean']:.4f} ± {summary['auc']['std']:.4f}")
print(f"F1 (burned)  : {summary['f1']['mean']:.4f} ± {summary['f1']['std']:.4f}")
print(f"Pearson — severity     : "
      f"{summary['severity_pearson']['mean']:.3f} ± "
      f"{summary['severity_pearson']['std']:.3f}")
print(f"Pearson — persistence  : "
      f"{summary['persistence_pearson']['mean']:.3f} ± "
      f"{summary['persistence_pearson']['std']:.3f}")
print(f"Pearson — recovery_gap : "
      f"{summary['recovery_gap_pearson']['mean']:.3f} ± "
      f"{summary['recovery_gap_pearson']['std']:.3f}")

## Experiment 2 — Spatial split (east/west)

Split the ROI by longitude. The west half historically has higher
fire activity; the east half has fewer fires. A model that overfits
to spatial autocorrelation will fail this holdout.

In [ ]:
EXP_NAME = "exp2_spatial"
EXP_DIR  = os.path.join(EXP_ROOT, EXP_NAME)

pixel_ids_kept = meta_df["pixel_id"].values
coords_kept    = coords_df.set_index("pixel_id").loc[pixel_ids_kept]
pixel_lon      = coords_kept["lon"].values
assert len(pixel_lon) == N, f"Mismatch: {len(pixel_lon)} vs {N}"

lon_median = np.median(pixel_lon)
is_west    = pixel_lon < lon_median

idx_west = np.where(is_west)[0]
idx_east = np.where(~is_west)[0]
print(f"West (train+val): {len(idx_west)}")
print(f"East (test):      {len(idx_east)}")

rng = np.random.RandomState(config.SEED)
rng.shuffle(idx_west)
n_val = int(len(idx_west) * 0.176)
spatial_val_idx   = idx_west[:n_val]
spatial_train_idx = idx_west[n_val:]

res_spatial = train_and_evaluate(
    spatial_train_idx, spatial_val_idx, idx_east,
    exp_name=EXP_NAME, exp_dir=EXP_DIR,
)

## Setup for window-based experiments (3 and 4)

These experiments work on time windows rather than the full series.
We recompute severity, persistence, and recovery_gap inside each
window so predictions and targets are aligned; recurrence is defined
by the pixel's whole history (a pixel's fire count is a fixed
attribute of the pixel).

In [ ]:
def compute_targets_window(X_full, y_full, nbr_raw_full, feature_names,
                           t_start, t_end,
                           baseline_months=12, recent_months=12):
    """Compute targets using only months [t_start, t_end)."""
    N, _, _ = X_full.shape
    try:
        nbr_z_idx = feature_names.index("NBR_zscore")
    except ValueError:
        nbr_z_idx = feature_names.index("NBR")

    severity     = np.zeros(N, dtype=np.float32)
    persistence  = np.zeros(N, dtype=np.float32)
    recovery_gap = np.zeros(N, dtype=np.float32)
    recurrence   = np.zeros(N, dtype=np.float32)

    for i in range(N):
        nbr_z         = X_full[i, t_start:t_end, nbr_z_idx]
        nbr_raw_win   = nbr_raw_full[i, t_start:t_end]
        y_win         = y_full[i, t_start:t_end]
        burned_months = np.where(y_win == 1)[0]

        recurrence[i] = float(np.clip(len(burned_months) / 5.0, 0, 1))

        if len(nbr_raw_win) >= baseline_months + recent_months:
            baseline_nbr = float(np.nanmean(nbr_raw_win[:baseline_months]))
            recent_nbr   = float(np.nanmean(nbr_raw_win[-recent_months:]))
            gap = (baseline_nbr - recent_nbr) / (abs(baseline_nbr) + 1e-6)
            recovery_gap[i] = float(np.clip(gap, 0, 1))

        if len(burned_months) == 0:
            continue
        drops       = -nbr_z[burned_months]
        severity[i] = float(np.clip(drops.mean() / 5.0, 0, 1))

        last_fire = burned_months[-1]
        post_fire = nbr_z[last_fire:]
        if len(post_fire) > 1:
            persistence[i] = float((post_fire < -0.5).mean())

    return {
        "recurrence":   recurrence,
        "severity":     severity,
        "persistence":  persistence,
        "recovery_gap": recovery_gap,
    }

In [ ]:
# Load only the four columns needed for window-specific recovery_gap. Explicit
# dtypes and vectorized indexing keep this under roughly 1 GB instead of loading
# and sorting a second copy of the full 36 GB CSV.
month_list_temp = config.get_month_list()
month_index = {(year, month): i
               for i, (year, month) in enumerate(month_list_temp)}

df_raw = pd.read_csv(
    config.FILES["raw_timeseries"],
    usecols=["pixel_id", "year", "month", "NBR"],
    dtype={"pixel_id": "int32", "year": "int16", "month": "int8",
           "NBR": "float32"},
)

pixel_ids = meta_df["pixel_id"].to_numpy(dtype=np.int32)
if np.any(pixel_ids[1:] < pixel_ids[:-1]):
    raise ValueError("Expected metadata pixel_id values to be sorted")

_raw_pixel_ids = df_raw["pixel_id"].to_numpy(dtype=np.int32, copy=False)
_i_of_row = np.searchsorted(pixel_ids, _raw_pixel_ids).astype(np.int32)
_linear_months = (df_raw["year"].to_numpy(dtype=np.int32) * 12 +
                  df_raw["month"].to_numpy(dtype=np.int32) - 1)
_month_keys = np.array(
    [year * 12 + month - 1 for year, month in month_list_temp],
    dtype=np.int32)
_month_low = int(_month_keys.min())
_month_high = int(_month_keys.max())
_month_lut = np.full(_month_high - _month_low + 1, -1, dtype=np.int16)
_month_lut[_month_keys - _month_low] = np.arange(T, dtype=np.int16)
_t_of_row = np.full(len(df_raw), -1, dtype=np.int16)
_in_range = ((_linear_months >= _month_low) &
             (_linear_months <= _month_high))
_t_of_row[_in_range] = _month_lut[
    _linear_months[_in_range] - _month_low]

_row_ok = (_t_of_row >= 0) & (_i_of_row < N)
_candidate_rows = np.flatnonzero(_row_ok)
_row_ok[_candidate_rows] &= (
    pixel_ids[_i_of_row[_candidate_rows]] == _raw_pixel_ids[_candidate_rows])

nbr_raw = np.full((N, T), np.nan, dtype=np.float32)
nbr_raw[_i_of_row[_row_ok], _t_of_row[_row_ok]] = (
    df_raw["NBR"].to_numpy(dtype=np.float32, copy=False)[_row_ok])
del (df_raw, _raw_pixel_ids, _i_of_row, _linear_months, _month_keys,
     _month_lut, _t_of_row, _in_range, _candidate_rows, _row_ok)

calendar_month = np.array([month for _, month in month_list_temp])
for cm in range(1, 13):
    t_idx = np.where(calendar_month == cm)[0]
    if len(t_idx) == 0:
        continue
    block = nbr_raw[:, t_idx]
    col_mean = np.nanmean(block)
    block[np.isnan(block)] = col_mean
    nbr_raw[:, t_idx] = block

print(f"NBR raw shape: {nbr_raw.shape}")
print(f"NBR raw range: [{nbr_raw.min():.3f}, {nbr_raw.max():.3f}]")


## Experiment 3 — Temporal split (window-based)

Three non-overlapping 36-month windows: 2017-2019, 2020-2022,
2023-2025. compute_targets_window recomputes analytical targets
inside each window so the prediction task is identical across
phases; only the underlying dynamics differ.

In [ ]:
month_list  = config.get_month_list()
year_of_t   = np.array([y for y, _ in month_list])

EXP_NAME = "exp3_temporal"
EXP_DIR  = os.path.join(EXP_ROOT, EXP_NAME)

TRAIN_START, TRAIN_END_TENT = 0,  36
VAL_START,   VAL_END_TENT   = 36, 72
TEST_START,  TEST_END_TENT  = 72, T

WINDOW_LEN = min(TRAIN_END_TENT - TRAIN_START,
                 VAL_END_TENT   - VAL_START,
                 TEST_END_TENT  - TEST_START)
TRAIN_END = TRAIN_START + WINDOW_LEN
VAL_END   = VAL_START   + WINDOW_LEN
TEST_END  = TEST_START  + WINDOW_LEN

assert TEST_END <= T
assert VAL_START  >= TRAIN_END
assert TEST_START >= VAL_END

print(f"Window length: {WINDOW_LEN} months")
print(f"Train window: months {TRAIN_START}..{TRAIN_END-1}  "
      f"({year_of_t[TRAIN_START]}-{year_of_t[TRAIN_END-1]})")
print(f"Val window:   months {VAL_START}..{VAL_END-1}  "
      f"({year_of_t[VAL_START]}-{year_of_t[VAL_END-1]})")
print(f"Test window:  months {TEST_START}..{TEST_END-1}  "
      f"({year_of_t[TEST_START]}-{year_of_t[TEST_END-1]})")

In [ ]:
def slice_window(X_full, y_full, mask_full, nbr_raw_full,
                 t_start, t_end, feature_names):
    # Views, not copies: basic slicing shares X's buffer, and nothing
    # writes to these arrays. .copy() here cost 4.6 GB per window (13.7 GB
    # across the three) duplicating data already resident in X.
    X_win    = X_full[:, t_start:t_end, :]
    y_win    = y_full[:, t_start:t_end]
    mask_win = mask_full[:, t_start:t_end]

    tgt = compute_targets_window(
        X_full, y_full, nbr_raw_full, feature_names, t_start, t_end)
    return X_win, y_win, mask_win, tgt


X_train_win, y_train_win, mask_train_win, tgt_train_win = slice_window(
    X, y, mask, nbr_raw, TRAIN_START, TRAIN_END, feature_names)
X_val_win, y_val_win, mask_val_win, tgt_val_win = slice_window(
    X, y, mask, nbr_raw, VAL_START, VAL_END, feature_names)
X_test_win, y_test_win, mask_test_win, tgt_test_win = slice_window(
    X, y, mask, nbr_raw, TEST_START, TEST_END, feature_names)

print(f"\nWindow tensor shape: {X_train_win.shape}")
print(f"\nFire prevalence by window:")
print(f"  Train  ({year_of_t[TRAIN_START]}-{year_of_t[TRAIN_END-1]}): "
      f"{int(y_train_win.sum())} burned months "
      f"({100*y_train_win.mean():.2f}%)")
print(f"  Val    ({year_of_t[VAL_START]}-{year_of_t[VAL_END-1]}): "
      f"{int(y_val_win.sum())} burned months "
      f"({100*y_val_win.mean():.2f}%)")
print(f"  Test   ({year_of_t[TEST_START]}-{year_of_t[TEST_END-1]}): "
      f"{int(y_test_win.sum())} burned months "
      f"({100*y_test_win.mean():.2f}%)")

In [ ]:
all_pixels = np.arange(N)


def make_window_loader(X_win, y_win, mask_win, tgt_win, indices,
                       batch_size, shuffle):
    """Window loader.

    This used to build TensorDataset(torch.tensor(X_win[indices]), ...), which
    materialised each split twice -- ~9.1 GB per loader. With three loaders in
    exp3 and three more in exp4, none ever freed, that retained tens of GB and
    OOM-killed the kernel once exp5 started. metrics.make_loader gathers each
    batch on demand instead; batch composition is identical.
    """
    return metrics.make_loader(X_win, y_win, mask_win, tgt_win, indices,
                               batch_size, shuffle)


train_loader_w = make_window_loader(
    X_train_win, y_train_win, mask_train_win, tgt_train_win,
    all_pixels, config.TRAINING["batch_size"], True)
val_loader_w = make_window_loader(
    X_val_win, y_val_win, mask_val_win, tgt_val_win,
    all_pixels, config.TRAINING["batch_size"], False)
test_loader_w = make_window_loader(
    X_test_win, y_test_win, mask_test_win, tgt_test_win,
    all_pixels, config.TRAINING["batch_size"], False)

In [ ]:
config.set_seed()
model_w = model_defs.FireRecoveryModel(
    n_features=C,
    d_model=config.MODEL["d_model"],
    n_head=config.MODEL["n_head"],
    d_k=config.MODEL["d_k"],
    max_len=WINDOW_LEN,
    tau=1000,
    dropout=config.MODEL["dropout"],
).to(DEVICE)

criterion_w = model_defs.MaskedMultiTaskLoss(
    alpha=config.LOSS["focal_alpha"],
    gamma=config.LOSS["focal_gamma"],
    lambda_aux=config.LOSS["lambda_aux"])
optimizer_w = optim.AdamW(model_w.parameters(),
                          lr=config.TRAINING["lr"],
                          weight_decay=config.TRAINING["weight_decay"])
scheduler_w = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_w, T_max=config.TRAINING["epochs"])

history_w = {"train": [], "val": []}
best_val_auc_w = -1.0
best_state_w   = None

_t0 = time.time()
for epoch in range(1, config.TRAINING["epochs"] + 1):
    _e0 = time.time()
    train_m = metrics.run_epoch(model_w, criterion_w, optimizer_w,
                                train_loader_w, DEVICE, train=True,
                                grad_clip=config.TRAINING["grad_clip"],
                                compute_auc=False)
    val_m   = metrics.run_epoch(model_w, criterion_w, optimizer_w,
                                val_loader_w, DEVICE, train=False)
    scheduler_w.step()
    history_w["train"].append(train_m)
    history_w["val"].append(val_m)
    if val_m["auc"] > best_val_auc_w:
        best_val_auc_w = val_m["auc"]
        best_state_w   = {k: v.cpu().clone()
                          for k, v in model_w.state_dict().items()}
    log_epoch("Exp3 temporal", epoch, config.TRAINING["epochs"], val_m,
              time.time() - _e0, _t0)

print(f"\nBest val AUC: {best_val_auc_w:.4f}")

In [ ]:
model_w.load_state_dict(best_state_w)
best_thr_w, _, _, _, _ = metrics.tune_threshold(model_w, val_loader_w, DEVICE)
burned_metrics_t, correlations_t, predictions_t = metrics.evaluate_test_set(
    model_w, test_loader_w, DEVICE, best_thr_w, tgt_test_win, all_pixels)
per_category_t = metrics.per_category_stats(
    predictions_t, all_pixels, meta_df, config.PIXEL_TYPES)
per_lc_t = metrics.per_lc_stats(
    predictions_t, all_pixels, lc_labels_all, config.LC_CATEGORIES)
n_params = sum(p.numel() for p in model_w.parameters())

metrics.save_experiment_results(
    EXP_DIR, EXP_NAME, best_state_w, history_w,
    burned_metrics_t, correlations_t, per_category_t,
    extra={
        "training":   {"best_val_auc": float(best_val_auc_w),
                       "n_epochs": config.TRAINING["epochs"]},
        "model_info": {"n_parameters": n_params,
                       "window_length": WINDOW_LEN,
                       "positional_tau": 1000,
                       "loss_type": "MaskedMultiTaskLoss"},
        "data_info":  {
            "train_window": [int(TRAIN_START), int(TRAIN_END)],
            "val_window":   [int(VAL_START),   int(VAL_END)],
            "test_window":  [int(TEST_START),  int(TEST_END)],
            "train_fires":  int(y_train_win.sum()),
            "val_fires":    int(y_val_win.sum()),
            "test_fires":   int(y_test_win.sum()),
        },
        "per_lc": per_lc_t,
    })
metrics.print_summary(EXP_NAME, n_params, burned_metrics_t,
                      correlations_t, per_category_t, config.PIXEL_TYPES)
metrics.print_lc_summary(per_lc_t, config.LC_CATEGORIES)
res_temporal = {"burned_metrics": burned_metrics_t,
                "correlations":   correlations_t,
                "per_category":   per_category_t,
                "per_lc":         per_lc_t,
                "best_val_auc":   best_val_auc_w}

## Experiment 4 — Combined spatial+temporal split (window-based)

Train on west + first window, val on west + middle window, test on
east + final window. The most demanding evaluation in this suite:
tests generalisation to unseen places at unseen times.

In [ ]:
EXP_NAME = "exp4_combined"
EXP_DIR  = os.path.join(EXP_ROOT, EXP_NAME)

idx_west_combined = idx_west
idx_east_combined = idx_east

print(f"Train (west, {year_of_t[TRAIN_START]}-{year_of_t[TRAIN_END-1]}): "
      f"{len(idx_west_combined)} pixels")
print(f"Val   (west, {year_of_t[VAL_START]}-{year_of_t[VAL_END-1]}):   "
      f"{len(idx_west_combined)} pixels")
print(f"Test  (east, {year_of_t[TEST_START]}-{year_of_t[TEST_END-1]}):   "
      f"{len(idx_east_combined)} pixels")

train_loader_c = make_window_loader(
    X_train_win, y_train_win, mask_train_win, tgt_train_win,
    idx_west_combined, config.TRAINING["batch_size"], True)
val_loader_c   = make_window_loader(
    X_val_win, y_val_win, mask_val_win, tgt_val_win,
    idx_west_combined, config.TRAINING["batch_size"], False)
test_loader_c  = make_window_loader(
    X_test_win, y_test_win, mask_test_win, tgt_test_win,
    idx_east_combined, config.TRAINING["batch_size"], False)

In [ ]:
config.set_seed()
model_c = model_defs.FireRecoveryModel(
    n_features=C, d_model=config.MODEL["d_model"],
    n_head=config.MODEL["n_head"], d_k=config.MODEL["d_k"],
    max_len=WINDOW_LEN, tau=1000,
    dropout=config.MODEL["dropout"],
).to(DEVICE)

criterion_c = model_defs.MaskedMultiTaskLoss(
    alpha=config.LOSS["focal_alpha"], gamma=config.LOSS["focal_gamma"],
    lambda_aux=config.LOSS["lambda_aux"])
optimizer_c = optim.AdamW(model_c.parameters(),
                          lr=config.TRAINING["lr"],
                          weight_decay=config.TRAINING["weight_decay"])
scheduler_c = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_c, T_max=config.TRAINING["epochs"])

history_c = {"train": [], "val": []}
best_val_auc_c = -1.0
best_state_c   = None

_t0 = time.time()
for epoch in range(1, config.TRAINING["epochs"] + 1):
    _e0 = time.time()
    train_m = metrics.run_epoch(model_c, criterion_c, optimizer_c,
                                train_loader_c, DEVICE, train=True,
                                grad_clip=config.TRAINING["grad_clip"],
                                compute_auc=False)
    val_m   = metrics.run_epoch(model_c, criterion_c, optimizer_c,
                                val_loader_c, DEVICE, train=False)
    scheduler_c.step()
    history_c["train"].append(train_m)
    history_c["val"].append(val_m)
    if val_m["auc"] > best_val_auc_c:
        best_val_auc_c = val_m["auc"]
        best_state_c   = {k: v.cpu().clone()
                          for k, v in model_c.state_dict().items()}
    log_epoch("Exp4 combined", epoch, config.TRAINING["epochs"], val_m,
              time.time() - _e0, _t0)

model_c.load_state_dict(best_state_c)
best_thr_c, _, _, _, _ = metrics.tune_threshold(model_c, val_loader_c, DEVICE)
burned_metrics_c, correlations_c, predictions_c = metrics.evaluate_test_set(
    model_c, test_loader_c, DEVICE, best_thr_c,
    tgt_test_win, idx_east_combined)
per_category_c = metrics.per_category_stats(
    predictions_c, idx_east_combined, meta_df, config.PIXEL_TYPES)
per_lc_c = metrics.per_lc_stats(
    predictions_c, idx_east_combined, lc_labels_all, config.LC_CATEGORIES)
n_params_c = sum(p.numel() for p in model_c.parameters())

metrics.save_experiment_results(
    EXP_DIR, EXP_NAME, best_state_c, history_c,
    burned_metrics_c, correlations_c, per_category_c,
    extra={"training":   {"best_val_auc": float(best_val_auc_c),
                          "n_epochs": config.TRAINING["epochs"]},
           "model_info": {"n_parameters": n_params_c,
                          "window_length": WINDOW_LEN,
                          "positional_tau": 1000,
                          "loss_type": "MaskedMultiTaskLoss"},
           "per_lc": per_lc_c})
metrics.print_summary(EXP_NAME, n_params_c, burned_metrics_c,
                      correlations_c, per_category_c, config.PIXEL_TYPES)
metrics.print_lc_summary(per_lc_c, config.LC_CATEGORIES)
res_combined = {"burned_metrics": burned_metrics_c,
                "correlations":   correlations_c,
                "per_category":   per_category_c,
                "per_lc":         per_lc_c,
                "best_val_auc":   best_val_auc_c}

## Experiment 5 — Random Forest baselines

Two RF baselines provide a sanity check on whether the L-TAE
architecture is justified. RF predicts the four regression targets
directly; per-month burned classification is skipped (RF is unlikely
to beat Focal on such an imbalanced problem).

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import PCA
from scipy.stats import pearsonr, spearmanr

with open(config.FILES["splits"]) as f:
    splits = json.load(f)
idx_train_rf = np.array(splits["train"])
idx_test_rf  = np.array(splits["test"])

X_flat = np.concatenate([X.mean(axis=1), X.std(axis=1)], axis=1)
print(f"Flattened features: {X_flat.shape[1]}")

In [ ]:
def evaluate_rf(X_in, name):
    """Train and evaluate one Random Forest per regression dimension."""
    correlations = {}
    for dim_name in ["severity", "persistence", "recovery_gap"]:
        rf = RandomForestRegressor(
            n_estimators=200,
            max_depth=None,
            n_jobs=-1,
            random_state=config.SEED,
        )
        rf.fit(X_in[idx_train_rf], targets[dim_name][idx_train_rf])
        preds = rf.predict(X_in[idx_test_rf])
        true_vals = targets[dim_name][idx_test_rf]
        r_p, _ = pearsonr(preds, true_vals)
        r_s, _ = spearmanr(preds, true_vals)
        correlations[dim_name] = {"pearson":  float(r_p),
                                  "spearman": float(r_s)}
        print(f"  {name} — {dim_name:15s}  Pearson={r_p:.3f}  Spearman={r_s:.3f}")
    return correlations


print("Random Forest on raw time-flattened features:")
rf_raw_results = evaluate_rf(X_flat, "RF raw")

pca = PCA(n_components=0.95, random_state=config.SEED)
X_flat_pca = pca.fit_transform(X_flat[idx_train_rf])
X_flat_pca_all = pca.transform(X_flat)
print(f"\nPCA: kept {X_flat_pca.shape[1]} components "
      f"(of {X_flat.shape[1]}), explaining "
      f"{pca.explained_variance_ratio_.sum() * 100:.1f}% variance")

print("\nRandom Forest on PCA-reduced features:")
rf_pca_results = evaluate_rf(X_flat_pca_all, "RF PCA")

EXP_NAME = "exp5_baselines"
EXP_DIR  = os.path.join(EXP_ROOT, EXP_NAME)
os.makedirs(EXP_DIR, exist_ok=True)
with open(os.path.join(EXP_DIR, "results.json"), "w") as f:
    json.dump({
        "rf_raw": {"correlations": rf_raw_results,
                   "n_features":   X_flat.shape[1]},
        "rf_pca": {"correlations": rf_pca_results,
                   "n_components": int(X_flat_pca.shape[1]),
                   "explained_variance":
                       float(pca.explained_variance_ratio_.sum())},
    }, f, indent=2)

## Experiment 6 — L-TAE on PCA features

Apply PCA across the feature dimension at each time step, preserving
temporal structure. Train L-TAE on the reduced features and compare
to the main model.

In [ ]:
EXP_NAME = "exp6_ltae_pca"
EXP_DIR  = os.path.join(EXP_ROOT, EXP_NAME)

# Compute the training-set feature covariance in chunks. PCA only needs this
# CxC matrix; selecting every training timestep would create another ~10 GB.
_pca_chunk_pixels = 10_000
_pca_sum = np.zeros(C, dtype=np.float64)
_pca_cross = np.zeros((C, C), dtype=np.float64)
_pca_count = 0

for _start in range(0, len(idx_train_rf), _pca_chunk_pixels):
    _train_pixels = idx_train_rf[_start:_start + _pca_chunk_pixels]
    _flat = X[_train_pixels].reshape(-1, C)
    _pca_sum += np.sum(_flat, axis=0, dtype=np.float64)
    _pca_cross += _flat.T @ _flat
    _pca_count += len(_flat)

_pca_mean = _pca_sum / _pca_count
_pca_cov = (_pca_cross - np.outer(_pca_sum, _pca_sum) / _pca_count)
_pca_cov /= (_pca_count - 1)
_eigenvalues, _eigenvectors = np.linalg.eigh(_pca_cov)
_order = np.argsort(_eigenvalues)[::-1]
_eigenvalues = np.maximum(_eigenvalues[_order], 0.0)
_eigenvectors = _eigenvectors[:, _order]
_explained_ratio = _eigenvalues / _eigenvalues.sum()
n_components_t = int(np.searchsorted(
    np.cumsum(_explained_ratio), 0.95) + 1)
_pca_components = _eigenvectors[:, :n_components_t].T.astype(np.float32)
_pca_mean = _pca_mean.astype(np.float32)

n_head = config.MODEL["n_head"]
pad_to = ((n_components_t + n_head - 1) // n_head) * n_head
X_pca_per_t = np.zeros((N, T, pad_to), dtype=np.float32)

# Transform contiguous pixel chunks directly into the final padded tensor.
for _start in range(0, N, _pca_chunk_pixels):
    _end = min(_start + _pca_chunk_pixels, N)
    _centered = X[_start:_end].reshape(-1, C) - _pca_mean
    _transformed = _centered @ _pca_components.T
    X_pca_per_t[_start:_end, :, :n_components_t] = _transformed.reshape(
        _end - _start, T, n_components_t)

print(f"PCA across feature dim: {C} -> {n_components_t} components "
      f"({_explained_ratio[:n_components_t].sum() * 100:.1f}% variance)")
if pad_to != n_components_t:
    print(f"Padded to {pad_to} channels for n_head={n_head} divisibility")

del (_train_pixels, _flat, _pca_cross, _pca_sum, _pca_cov,
     _eigenvectors, _centered, _transformed)


In [ ]:
import metrics as _metrics
orig_make_loader = _metrics.make_loader


def make_loader_pca(X_arg, *args, **kwargs):
    return orig_make_loader(X_pca_per_t, *args, **kwargs)


_metrics.make_loader = make_loader_pca
idx_val_rf = np.array(splits["val"])

try:
    res_ltae_pca = train_and_evaluate(
        idx_train_rf, idx_val_rf, idx_test_rf,
        exp_name=EXP_NAME, exp_dir=EXP_DIR,
        n_features_override=X_pca_per_t.shape[2],
    )
finally:
    _metrics.make_loader = orig_make_loader

# Later ablations use the original X, which was never overwritten.
del X_pca_per_t


## Experiment 7 — Architecture ablations

Four variations, each isolating one design choice: positional
encoding, single head, eight heads, no final MLP.

In [ ]:
EXP_NAME_ROOT = "exp7_ablations"
EXP_DIR_ROOT  = os.path.join(EXP_ROOT, EXP_NAME_ROOT)
os.makedirs(EXP_DIR_ROOT, exist_ok=True)

with open(config.FILES["splits"]) as f:
    splits_main = json.load(f)
abl_train_idx = np.array(splits_main["train"])
abl_val_idx   = np.array(splits_main["val"])
abl_test_idx  = np.array(splits_main["test"])


def run_ablation(label, override):
    """Train a model with one hyperparameter swap and evaluate it."""
    config.set_seed()
    cfg = {
        "d_model":     config.MODEL["d_model"],
        "n_head":      config.MODEL["n_head"],
        "d_k":         config.MODEL["d_k"],
        "tau":         config.MODEL["positional_tau"],
        "dropout":     config.MODEL["dropout"],
        "use_pos_enc": True,
        "mlp_dims":    [config.MODEL["d_model"]],
    }
    cfg.update(override)

    model_abl = model_defs.FireRecoveryModel(
        n_features=C,
        d_model=cfg["d_model"],
        n_head=cfg["n_head"],
        d_k=cfg["d_k"],
        max_len=T,
        tau=cfg["tau"],
        dropout=cfg["dropout"],
    ).to(DEVICE)

    if not cfg["use_pos_enc"]:
        class _NoPosEnc(torch.nn.Module):
            def forward(self, x): return x
        model_abl.ltae.pos_enc = _NoPosEnc().to(DEVICE)

    if cfg["mlp_dims"] is None:
        model_abl.ltae.mlp = torch.nn.Identity().to(DEVICE)

    train_loader = metrics.make_loader(
        X, y, mask, targets, abl_train_idx,
        config.TRAINING["batch_size"], True)
    val_loader   = metrics.make_loader(
        X, y, mask, targets, abl_val_idx,
        config.TRAINING["batch_size"], False)
    test_loader  = metrics.make_loader(
        X, y, mask, targets, abl_test_idx,
        config.TRAINING["batch_size"], False)

    criterion = model_defs.MaskedMultiTaskLoss(
        alpha=config.LOSS["focal_alpha"],
        gamma=config.LOSS["focal_gamma"],
        lambda_aux=config.LOSS["lambda_aux"])
    optimizer = optim.AdamW(model_abl.parameters(),
                            lr=config.TRAINING["lr"],
                            weight_decay=config.TRAINING["weight_decay"])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config.TRAINING["epochs"])

    history = {"train": [], "val": []}
    best_val_auc = -1.0
    best_state   = None

    _t0 = time.time()
    for epoch in range(1, config.TRAINING["epochs"] + 1):
        _e0 = time.time()
        train_m = metrics.run_epoch(model_abl, criterion, optimizer,
                                    train_loader, DEVICE, train=True,
                                    grad_clip=config.TRAINING["grad_clip"],
                                    compute_auc=False)
        val_m   = metrics.run_epoch(model_abl, criterion, optimizer,
                                    val_loader, DEVICE, train=False)
        scheduler.step()
        history["train"].append(train_m)
        history["val"].append(val_m)
        log_epoch(f"Exp7 {label}", epoch, config.TRAINING["epochs"], val_m,
                  time.time() - _e0, _t0, every=5)
        if val_m["auc"] > best_val_auc:
            best_val_auc = val_m["auc"]
            best_state   = {k: v.cpu().clone()
                            for k, v in model_abl.state_dict().items()}

    model_abl.load_state_dict(best_state)
    best_thr, _, _, _, _ = metrics.tune_threshold(
        model_abl, val_loader, DEVICE)
    burned_metrics, correlations, predictions = metrics.evaluate_test_set(
        model_abl, test_loader, DEVICE, best_thr, targets, abl_test_idx)
    per_category = metrics.per_category_stats(
        predictions, abl_test_idx, meta_df, config.PIXEL_TYPES)
    per_lc = metrics.per_lc_stats(
        predictions, abl_test_idx, lc_labels_all, config.LC_CATEGORIES)

    n_params = sum(p.numel() for p in model_abl.parameters())
    exp_dir = os.path.join(EXP_DIR_ROOT, label)
    metrics.save_experiment_results(
        exp_dir, f"{EXP_NAME_ROOT}_{label}", best_state, history,
        burned_metrics, correlations, per_category,
        extra={"training":   {"best_val_auc": float(best_val_auc),
                              "n_epochs":     config.TRAINING["epochs"]},
               "model_info": {"n_parameters": n_params,
                              "ablation":     cfg,
                              "loss_type":    "MaskedMultiTaskLoss"},
               "per_lc":     per_lc})
    metrics.print_summary(label, n_params, burned_metrics,
                          correlations, per_category, config.PIXEL_TYPES)
    return {"burned_metrics": burned_metrics,
            "correlations":   correlations,
            "per_category":   per_category,
            "per_lc":         per_lc,
            "n_params":       n_params}

In [ ]:
print("\n" + "#" * 70)
print("# ABLATION 1 — no positional encoding")
print("#" * 70)
res_abl_no_pos = run_ablation("no_positional_encoding", {"use_pos_enc": False})

In [ ]:
print("\n" + "#" * 70)
print("# ABLATION 2 — n_head = 1")
print("#" * 70)
res_abl_1head = run_ablation("n_head_1", {"n_head": 1})

In [ ]:
print("\n" + "#" * 70)
print("# ABLATION 3 — n_head = 8")
print("#" * 70)
res_abl_8head = run_ablation("n_head_8", {"n_head": 8})

In [ ]:
print("\n" + "#" * 70)
print("# ABLATION 4 — no final MLP")
print("#" * 70)
res_abl_no_mlp = run_ablation("no_final_mlp", {"mlp_dims": None})

## Final summary — comparison table

In [ ]:
def load_results(exp_name):
    path = os.path.join(EXP_ROOT, exp_name, "results.json")
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return None


with open(os.path.join(config.PATHS["models_main"], "results_main.json")) as f:
    main_results = json.load(f)

rows = []

rows.append({
    "experiment":  "Main (random split)",
    "AUC":         main_results["burned_detection"]["auc"],
    "F1":          main_results["burned_detection"]["f1"],
    "Pearson_sev": main_results["correlations"]["severity"]["pearson"],
    "Pearson_per": main_results["correlations"]["persistence"]["pearson"],
    "Pearson_rec": main_results["correlations"]["recovery_gap"]["pearson"],
})

exp1_sum_path = os.path.join(EXP_ROOT, "exp1_random_kfold", "summary.json")
if os.path.exists(exp1_sum_path):
    with open(exp1_sum_path) as f:
        s = json.load(f)
    rows.append({
        "experiment":  "5-Fold CV (mean ± std)",
        "AUC":         f"{s['auc']['mean']:.3f} ± {s['auc']['std']:.3f}",
        "F1":          f"{s['f1']['mean']:.3f} ± {s['f1']['std']:.3f}",
        "Pearson_sev":
            f"{s['severity_pearson']['mean']:.3f} ± "
            f"{s['severity_pearson']['std']:.3f}",
        "Pearson_per":
            f"{s['persistence_pearson']['mean']:.3f} ± "
            f"{s['persistence_pearson']['std']:.3f}",
        "Pearson_rec":
            f"{s['recovery_gap_pearson']['mean']:.3f} ± "
            f"{s['recovery_gap_pearson']['std']:.3f}",
    })

for exp_name, label in [
    ("exp2_spatial",  "Spatial split"),
    ("exp3_temporal", "Temporal split"),
    ("exp4_combined", "Combined spatial+temporal"),
    ("exp6_ltae_pca", "L-TAE on PCA features"),
]:
    r = load_results(exp_name)
    if r is None:
        continue
    rows.append({
        "experiment":  label,
        "AUC":         r["burned_detection"]["auc"],
        "F1":          r["burned_detection"]["f1"],
        "Pearson_sev": r["correlations"]["severity"]["pearson"],
        "Pearson_per": r["correlations"]["persistence"]["pearson"],
        "Pearson_rec": r["correlations"]["recovery_gap"]["pearson"],
    })

rf_results = load_results("exp5_baselines")
if rf_results:
    for k, label in [("rf_raw", "RF raw features"),
                     ("rf_pca", "RF PCA features")]:
        rows.append({
            "experiment":  label,
            "AUC":         "—",
            "F1":          "—",
            "Pearson_sev": rf_results[k]["correlations"]["severity"]["pearson"],
            "Pearson_per": rf_results[k]["correlations"]["persistence"]["pearson"],
            "Pearson_rec": rf_results[k]["correlations"]["recovery_gap"]["pearson"],
        })

for abl_name, abl_label in [
    ("no_positional_encoding", "Ablation: no pos. encoding"),
    ("n_head_1",                "Ablation: n_head=1"),
    ("n_head_8",                "Ablation: n_head=8"),
    ("no_final_mlp",            "Ablation: no final MLP"),
]:
    abl_path = os.path.join(EXP_ROOT, "exp7_ablations", abl_name, "results.json")
    if os.path.exists(abl_path):
        with open(abl_path) as f:
            r = json.load(f)
        rows.append({
            "experiment":  abl_label,
            "AUC":         r["burned_detection"]["auc"],
            "F1":          r["burned_detection"]["f1"],
            "Pearson_sev": r["correlations"]["severity"]["pearson"],
            "Pearson_per": r["correlations"]["persistence"]["pearson"],
            "Pearson_rec": r["correlations"]["recovery_gap"]["pearson"],
        })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

summary_path = os.path.join(config.PATHS["outputs_tab"], "experiment_summary.csv")
summary_df.to_csv(summary_path, index=False)
print(f"\nSummary written to: {summary_path}")

## Experiment 8 — Deployment realism check

The training sample is stratified so rare recurrence categories are
well-represented; the actual ROI is dominated by no_fires pixels.
Reweight test-set metrics to reflect the ROI distribution.

In [ ]:
import rasterio
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

EXP_NAME = "exp8_deployment_realism"
EXP_DIR  = os.path.join(EXP_ROOT, EXP_NAME)
os.makedirs(EXP_DIR, exist_ok=True)

with rasterio.open(config.FILES["burn_count_tif"]) as src:
    burn_count_raw = src.read(1)
    eco_mask       = src.read(2)

valid = (eco_mask == 1)
bc = burn_count_raw[valid]

roi_counts = {
    "no_fires":          int(np.sum(bc == 0)),
    "low_recurrence":    int(np.sum((bc >= 1) & (bc <= 2))),
    "medium_recurrence": int(np.sum((bc >= 3) & (bc <= 4))),
    "high_recurrence":   int(np.sum(bc >= 5)),
}
roi_total = sum(roi_counts.values())

print("Actual ROI distribution (eco-masked):")
for k, v in roi_counts.items():
    print(f"  {k:20s}  {v:>12,}  ({100 * v / roi_total:5.2f}%)")
print(f"  {'TOTAL':20s}  {roi_total:>12,}")

In [ ]:
train_counts = meta_df["pixel_type"].value_counts().to_dict()
train_total  = sum(train_counts.values())

print("\nStratified sample distribution (training data):")
for k in config.PIXEL_TYPES:
    n = train_counts.get(k, 0)
    print(f"  {k:20s}  {n:>5,}  ({100 * n / train_total:5.2f}%)")

weights_per_category = {}
for k in config.PIXEL_TYPES:
    roi_prop    = roi_counts[k] / roi_total
    sample_prop = train_counts.get(k, 1) / train_total
    weights_per_category[k] = roi_prop / sample_prop

print("\nWeights to align sample with ROI distribution:")
for k, w in weights_per_category.items():
    print(f"  {k:20s}  weight = {w:.4f}")

In [ ]:
preds_test = dict(np.load(config.FILES["preds_test"]))
with open(config.FILES["splits"]) as f:
    splits_main = json.load(f)
idx_test_main = np.array(splits_main["test"])

pixel_types_test = meta_df.iloc[idx_test_main]["pixel_type"].values
pixel_weights    = np.array([weights_per_category[pt]
                             for pt in pixel_types_test])

test_probs  = preds_test["burned_probs"].reshape(len(idx_test_main), -1)
test_labels = preds_test["burned_labels"].reshape(len(idx_test_main), -1)
test_mask   = preds_test["mask"].reshape(len(idx_test_main), -1).astype(bool)

month_weights = np.repeat(pixel_weights, test_probs.shape[1]).reshape(
    test_probs.shape)

In [ ]:
flat_probs   = test_probs.reshape(-1)
flat_labels  = test_labels.reshape(-1)
flat_mask    = test_mask.reshape(-1)
flat_weights = month_weights.reshape(-1)

m = flat_mask
probs_m   = flat_probs[m]
labels_m  = flat_labels[m]
weights_m = flat_weights[m]

auc_weighted = roc_auc_score(labels_m, probs_m, sample_weight=weights_m)
auc_unweight = roc_auc_score(labels_m, probs_m)

threshold = float(preds_test["threshold"])
preds_bin = (probs_m >= threshold).astype(int)

f1_weighted   = f1_score(labels_m, preds_bin,
                         sample_weight=weights_m, zero_division=0)
f1_unweight   = f1_score(labels_m, preds_bin, zero_division=0)
prec_weighted = precision_score(labels_m, preds_bin,
                                sample_weight=weights_m, zero_division=0)
rec_weighted  = recall_score(labels_m, preds_bin,
                             sample_weight=weights_m, zero_division=0)

print("Burned-month classification — weighted vs unweighted:")
print(f"  AUC weighted (ROI dist.):   {auc_weighted:.4f}")
print(f"  AUC unweighted (sample):    {auc_unweight:.4f}")
print(f"  F1  weighted (ROI dist.):   {f1_weighted:.4f}")
print(f"  F1  unweighted (sample):    {f1_unweight:.4f}")
print(f"  Precision weighted:         {prec_weighted:.4f}")
print(f"  Recall weighted:            {rec_weighted:.4f}")

In [ ]:
from scipy.stats import pearsonr, spearmanr


def weighted_pearson(x, y, w):
    """Weighted Pearson correlation."""
    w_sum = w.sum()
    x_mean = (w * x).sum() / w_sum
    y_mean = (w * y).sum() / w_sum
    cov_xy = (w * (x - x_mean) * (y - y_mean)).sum() / w_sum
    var_x  = (w * (x - x_mean) ** 2).sum() / w_sum
    var_y  = (w * (y - y_mean) ** 2).sum() / w_sum
    return cov_xy / np.sqrt(var_x * var_y)


targets_test = {
    "severity":     targets["severity"][idx_test_main],
    "persistence":  targets["persistence"][idx_test_main],
    "recovery_gap": targets["recovery_gap"][idx_test_main],
}
preds_per_pixel = {
    "severity":     preds_test["severity"],
    "persistence":  preds_test["persistence"],
    "recovery_gap": preds_test["recovery_gap"],
}

print("\nDimension correlations — weighted vs unweighted:")
print(f"  {'dimension':15s}  {'unweighted':>11s}  {'weighted':>10s}")
correlations_weighted = {}
for dim in ["severity", "persistence", "recovery_gap"]:
    r_uw, _ = pearsonr(preds_per_pixel[dim], targets_test[dim])
    r_w     = weighted_pearson(preds_per_pixel[dim], targets_test[dim],
                                pixel_weights)
    correlations_weighted[dim] = {
        "pearson_unweighted": float(r_uw),
        "pearson_weighted":   float(r_w),
    }
    print(f"  {dim:15s}  {r_uw:>11.4f}  {r_w:>10.4f}")

In [ ]:
results = {
    "experiment_name": EXP_NAME,
    "roi_distribution": {
        "counts": roi_counts,
        "total":  roi_total,
        "proportions": {k: v / roi_total for k, v in roi_counts.items()},
    },
    "sample_distribution": {
        "counts": train_counts,
        "total":  train_total,
    },
    "weights": weights_per_category,
    "weighted_metrics": {
        "burned": {
            "auc_unweighted":     float(auc_unweight),
            "auc_weighted":       float(auc_weighted),
            "f1_unweighted":      float(f1_unweight),
            "f1_weighted":        float(f1_weighted),
            "precision_weighted": float(prec_weighted),
            "recall_weighted":    float(rec_weighted),
        },
        "correlations": correlations_weighted,
    },
}

with open(os.path.join(EXP_DIR, "results.json"), "w") as f:
    json.dump(results, f, indent=2)

print(f"\nSaved to {EXP_DIR}/results.json")